World Bank - Country Lookup Table
=================================

In [1]:
from pathlib import Path

import pandas as pd
import wbgapi as wb

from utils.paths import STG_WORLD_BANK_DIR

In [ ]:
WB_REGIONS_FILENAME = 'stg_world_bank__regions.csv'
WB_REGIONS_FILEPATH = STG_WORLD_BANK_DIR / WB_REGIONS_FILENAME

'c:\\Users\\chink\\OneDrive - ISED-ISDE\\Documents\\Projects\\or_country_profiles\\data\\common-tables\\regions\\World_Bank-regions.csv'

In [3]:
wb_economies: pd.DataFrame = wb.economy.DataFrame()

In [4]:
print(wb_economies.info())
wb_economies.head(1)

<class 'pandas.core.frame.DataFrame'>
Index: 266 entries, ABW to ZWE
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   name         266 non-null    object 
 1   aggregate    266 non-null    bool   
 2   longitude    211 non-null    float64
 3   latitude     211 non-null    float64
 4   region       266 non-null    object 
 5   adminregion  266 non-null    object 
 6   lendingType  266 non-null    object 
 7   incomeLevel  266 non-null    object 
 8   capitalCity  266 non-null    object 
dtypes: bool(1), float64(2), object(6)
memory usage: 19.0+ KB
None


,name,aggregate,longitude,latitude,region,adminregion,lendingType,incomeLevel,capitalCity
id,,,,,,,,,
ABW,Aruba,False,-70.0167,12.5167,LCN,,LNX,HIC,Oranjestad


In [45]:
for col in wb_economies.columns:
    print(col)

name
aggregate
longitude
latitude
region
adminregion
lendingType
incomeLevel
capitalCity


In [31]:
wb_economies_df = (
    wb_economies
    .reset_index()
    .rename(columns={
        'id': 'wb_economy_code',
        'name': 'wb_economy_name',
        'region': 'region_code',
        'adminregion': 'admin_region_code'
    })
    .replace({'': pd.NA})
)

In [49]:
for col in wb_economies_df.columns:
    print(col)

wb_economy_code
wb_economy_name
aggregate
longitude
latitude
region_code
admin_region_code
lendingType
incomeLevel
capitalCity


In [9]:
wb_regions: pd.Series = wb.region.Series()  # region API call

In [32]:
# Create map of all World Bank economies to their
# respective regional groups (i.e. 'regions') 

wb_regions_df = (
    wb_regions
    .to_frame(name='region_name')
    .reset_index()
    .rename(columns={'index': 'region_code'})
)

wb_region_map_df = (
    wb_economies_df
    .loc[:, ['wb_economy_code', 'wb_economy_name', 'region_code']]
    .merge(
        wb_regions_df,
        on='region_code',
        how='inner'
    )
    .loc[:, [
        'region_code',
        'region_name',
        'wb_economy_code',
        'wb_economy_name'
    ]]
    .sort_values(by=['region_code', 'wb_economy_code'], ignore_index=True)
    .reset_index(drop=True)
)

In [33]:
wb_region_map_df

,region_code,region_name,wb_economy_code,wb_economy_name
0,EAS,East Asia & Pacific,ASM,American Samoa
1,EAS,East Asia & Pacific,AUS,Australia
2,EAS,East Asia & Pacific,BRN,Brunei Darussalam
3,EAS,East Asia & Pacific,CHN,China
4,EAS,East Asia & Pacific,FJI,Fiji
...,...,...,...,...
212,SSF,Sub-Saharan Africa,TZA,Tanzania
213,SSF,Sub-Saharan Africa,UGA,Uganda
214,SSF,Sub-Saharan Africa,ZAF,South Africa
215,SSF,Sub-Saharan Africa,ZMB,Zambia


In [43]:
wb_admin_region_map_df = (
    wb_economies_df
    .loc[:, ['wb_economy_code', 'wb_economy_name', 'admin_region_code']]
    .merge(
        wb_regions_df,
        left_on='admin_region_code',
        right_on='region_code',
        how='inner'
    )
    .rename(columns={
        'region_name': 'admin_region_name'
    })
    .loc[:, [
        'admin_region_code',
        'admin_region_name',
        'wb_economy_code',
        'wb_economy_name'
    ]]
)

In [44]:
wb_admin_region_map_df

,admin_region_code,admin_region_name,wb_economy_code,wb_economy_name
0,MNA,"Middle East, North Africa, Afghanistan & Pakis...",AFG,Afghanistan
1,SSA,Sub-Saharan Africa (excluding high income),AGO,Angola
2,ECA,Europe & Central Asia (excluding high income),ALB,Albania
3,LAC,Latin America & Caribbean (excluding high income),ARG,Argentina
4,ECA,Europe & Central Asia (excluding high income),ARM,Armenia
...,...,...,...,...
124,ECA,Europe & Central Asia (excluding high income),XKX,Kosovo
125,MNA,"Middle East, North Africa, Afghanistan & Pakis...",YEM,"Yemen, Rep."
126,SSA,Sub-Saharan Africa (excluding high income),ZAF,South Africa
127,SSA,Sub-Saharan Africa (excluding high income),ZMB,Zambia


In [ ]:
# TODO: Execute code below to join admin region name with wb_economies_df

wb_combined_df = (
    wb_economies_df
    .merge(
        wb_admin_region_map_df[['admin_region_code', 'admin_region_name']],
        left_on='admin_region',
        right_on='admin_region_code',
        how='left'
    )
    .drop(columns='admin_region')
) 

In [ ]:
wb_combined_df

In [ ]:
# Replace wb_economies_df if join w/ admin regions works as desired
wb_economies_df = wb_combined_df.copy()

In [ ]:
# Check that filename ends in '.csv', then write df to CSV
if WB_REGIONS_FILENAME.endswith('.csv'):
    wb_economies_df.to_csv(wb_regions_path, index=False)
    
    relative_dir = common_tables_dir.relative_to(project_dir)
    
    print(f"DataFrame written to file '{WB_REGIONS_FILENAME}'")
    print(f"Filepath: .\\{relative_dir}\\{WB_REGIONS_FILENAME}")
else:
    print("Abort write: filename missing '.csv' file extension")